# Credit Card Default Prediction

This notebook implements an end-to-end machine learning workflow
for predicting credit card default using a tabular dataset from
the UCI Machine Learning Repository.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
df = pd.read_excel("data/default-of-credit-card-clients.xls", header=1)

In [ ]:
df.head()
df.info
print("Shape:", df.shape)
print("null: ", df.isnull().sum().sum())
print("duplicate: ",df.duplicated().sum())
print("columns: ",df.columns)

In [ ]:
df = df.drop(columns=['ID'])

df.drop_duplicates(inplace=True)

df.rename(columns={'default payment next month': 'Default'}, inplace=True)


In [ ]:
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

In [ ]:
df['EDUCATION'] = df['EDUCATION'].replace({
    1: 'Grad School',
    2: 'University',
    3: 'High School',
    4: 'Other'
})


df['MARRIAGE'] = df['MARRIAGE'].replace({
    1: 'Married',
    2: 'Single',
    3: 'Other'
})

df['SEX'] = df['SEX'].replace({
    1: 'Male',
    2: 'Female'
})

In [ ]:
new_column_names = {

    'PAY_0': 'Pay_Sep',    
    'PAY_2': 'Pay_Aug',   
    'PAY_3': 'Pay_July',  
    'PAY_4': 'Pay_June',  
    'PAY_5': 'Pay_May',   
    'PAY_6': 'Pay_April', 
    
    'BILL_AMT1': 'Bill_Amt_Sep',
    'BILL_AMT2': 'Bill_Amt_Aug',
    'BILL_AMT3': 'Bill_Amt_July',
    'BILL_AMT4': 'Bill_Amt_June',
    'BILL_AMT5': 'Bill_Amt_May',
    'BILL_AMT6': 'Bill_Amt_April',

    'PAY_AMT1': 'Pay_Amt_Sep',
    'PAY_AMT2': 'Pay_Amt_Aug',
    'PAY_AMT3': 'Pay_Amt_July',
    'PAY_AMT4': 'Pay_Amt_June',
    'PAY_AMT5': 'Pay_Amt_May',
    'PAY_AMT6': 'Pay_Amt_April',

    'LIMIT_BAL': 'Limit_Bal',
    'AGE': 'Age',
    'EDUCATION': 'Education',
    'SEX': 'Sex',
    'MARRIAGE': 'Marriage'
}

df.rename(columns=new_column_names, inplace=True)


In [ ]:
categorical_cols = ['Sex', 'Education', 'Marriage']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
X = df.drop('Default', axis=1)  
y = df['Default']               
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    
    random_state=42,  
    stratify=y       
)

# Ölçeklenmesi gereken kolonları seç (0–1 dışında geniş aralığa sahip olanlar)
cols_to_scale = [col for col in X_train.columns if X_train[col].max() > 1]
scaler = StandardScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])



In [ ]:
# Let's examine the distribution of the target variable.
print(y_train.value_counts(normalize=True))

### Summary of Data Preprocessing Steps

In this section, we prepared the raw dataset for machine learning models through the following sequential steps:

1.  **Data Loading & Cleanup:**
    * Loaded the dataset and removed the `ID` column as it provides no predictive value.
    * Identified and removed **duplicate rows** to ensure data integrity and prevent the model from memorizing identical samples.

2.  **Feature Cleaning:**
    * Grouped unknown or undocumented values in `EDUCATION` and `MARRIAGE` columns into a single "Other" category to reduce noise.
    * Mapped numerical category codes (e.g., 1, 2) to readable string labels (e.g., "Graduate School", "Single") for better interpretability.

3.  **Renaming Columns:**
    * Renamed confusing column names (like `PAY_0`, `BILL_AMT1`) to clearer, time-based names (e.g., `Pay_Sep`, `Bill_Amt_Sep`) to easily track the monthly history.

4.  **Encoding:**
    * Applied **One-Hot Encoding** (using `get_dummies`) to convert categorical text variables (`Sex`, `Education`, `Marriage`) into a numerical format that models can understand.

5.  **Train-Test Split:**
    * Split the data into **Training (80%)** and **Testing (20%)** sets.
    * Used `stratify=y` to maintain the same proportion of default/non-default cases in both sets.

6.  **Feature Scaling:**
    * Applied **StandardScaler** to numerical columns (such as `Limit_Bal`, `Age`, `Bill_Amt`) to normalize their range.
    * **Important:** The scaler was fitted **only on the Training set** and then applied to the Test set to strictly avoid data leakage.
    